In [ ]:
# Copyright (c) Microsoft Corporation.
# Licensed under the MIT license.

# MtSi1005 pretraining montage examples

This notebook has two deliberately separate roles:

1. **Study gallery:** display the pre-rendered final QC images for all 12 pretraining datasets.
2. **Runnable method demo:** exercise the public `mtrenirs.alignment` API with small, package-authored synthetic arrays.

For ready-to-load synthetic files covering canonical JSON, 2D/3D tables, NPZ, BIDS fNIRS, and SNIRF, see [`../example_test_input/README.md`](../example_test_input/README.md). Use those files to exercise the UI and CLI import paths; use this notebook to understand the alignment methods and review the retained pretraining QC gallery.

> **Privacy boundary:** This notebook and the package contain no machine-readable geometry derived from the 12 pretraining datasets. The static gallery images are documentation artifacts and cannot reconstruct a source montage. All runnable arrays and files are package-authored synthetic data.

In [ ]:
from __future__ import annotations

import io
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from mtrenirs.alignment import align, align_optodes, load_reference


def find_final_results() -> Path:
    candidates = [
        Path.cwd() / "final-results",
        Path.cwd() / "examples_opensource_fnirs" / "final-results",
        Path.cwd().parent / "examples_opensource_fnirs" / "final-results",
    ]
    for candidate in candidates:
        if candidate.is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find examples_opensource_fnirs/final-results. Run this notebook from "
        "the repository root or examples_opensource_fnirs directory."
    )


FINAL_RESULTS = find_final_results()
print("Using bundled rendered examples from examples_opensource_fnirs/final-results")

## 1. The 12 pretraining montages

The metadata below describes only the public workflow boundary: source container, available geometry type, alignment strategy, and the corresponding rendered QC image. It intentionally omits reconstructive geometry and fixed mapping tables.

In [ ]:
DATASETS = [

    {"name": "Ajra2026", "source_format": "SNIRF", "geometry": "device template", "strategy": "fixed canonical labels", "public_input_schema": "channel names + canonical labels", "image": "montage_Ajra2026.png"},

    {"name": "Chen2023", "source_format": "MATLAB", "geometry": "paper placement", "strategy": "fixed canonical labels", "public_input_schema": "channel names + canonical labels", "image": "montage_Chen2023.png"},

    {"name": "Defenderfer2019", "source_format": "Homer .nirs", "geometry": "planar schematic", "strategy": "paper anchor + canonical 3D nearest", "public_input_schema": "adapter-placed channel names + (x, y, z)", "image": "montage_Defenderfer2019.png"},

    {"name": "Ferron2019", "source_format": "SNIRF + BIDS sidecar", "geometry": "digitized 3D", "strategy": "isotropic 3D nearest", "public_input_schema": "SNIRF or channel names + (x, y, z)", "image": "montage_Ferron2019.png"},

    {"name": "Gao2023", "source_format": "SNIRF + BIDS optodes.tsv", "geometry": "digitized 3D optodes", "strategy": "isotropic 3D nearest", "public_input_schema": "optode arrays + S-D pairs", "image": "montage_Gao2023-ballsqueezing.png"},

    {"name": "Hudak2017", "source_format": "signal text + PS1/PS2 montage text", "geometry": "MNI-derived labels", "strategy": "fixed canonical labels", "public_input_schema": "channel names + canonical labels", "image": "montage_Hudak2017-ADHD.png"},

    {"name": "Li2020", "source_format": "SNIRF + BIDS optodes.tsv", "geometry": "planar schematic", "strategy": "paper anchor + canonical 3D nearest", "public_input_schema": "adapter-placed channel names + (x, y, z)", "image": "montage_Li2020.png"},

    {"name": "Morgan2026", "source_format": "SNIRF + BIDS optodes.tsv", "geometry": "digitized 3D", "strategy": "isotropic 3D nearest", "public_input_schema": "optode arrays + S-D pairs", "image": "montage_Morgan2026.png"},

    {"name": "Shin2017", "source_format": "signal MAT + montage MAT", "geometry": "template 3D", "strategy": "isotropic 3D nearest", "public_input_schema": "montage channel names + (x, y, z)", "image": "montage_Shin2017.png"},

    {"name": "Spape2023", "source_format": "SNIRF + BIDS optodes.tsv", "geometry": "planar schematic", "strategy": "paper anchor + canonical 3D nearest", "public_input_schema": "adapter-placed channel names + (x, y, z)", "image": "montage_Spape2023-emotion.png"},

    {"name": "Sudan2026", "source_format": "SNIRF", "geometry": "digitized 3D, non-standard axes", "strategy": "axis correction + isotropic 3D", "public_input_schema": "adapter-corrected names + (x, y, z)", "image": "montage_Sudan2026-cocktail-visual.png"},

    {"name": "Xiafei2024", "source_format": "SNIRF + BIDS optodes.tsv", "geometry": "device template", "strategy": "fixed canonical labels", "public_input_schema": "channel names + canonical labels", "image": "montage_Xiafei2024-driving.png"},

]



assert len(DATASETS) == 12

for dataset in DATASETS:

    print(f"{dataset['name']:18} | {dataset['strategy']:38} | {dataset['source_format']}")

## 2. Synthetic input records

These small records illustrate the public schemas. They are authored for this notebook and are not measurements, approximations, or transformations of the 12 study montages.

In [ ]:
SYNTHETIC_3D_ROWS = [
    {"name": "demo01", "x": -31.0, "y": 78.0, "z": 52.0},
    {"name": "demo02", "x": 30.0, "y": 79.0, "z": 51.0},
    {"name": "demo03", "x": -42.0, "y": 5.0, "z": 76.0},
    {"name": "demo04", "x": 41.0, "y": 6.0, "z": 75.0},
]
SYNTHETIC_2D_ROWS = [
    {"name": "grid01", "x": -30.0, "y": 20.0},
    {"name": "grid02", "x": -10.0, "y": 30.0},
    {"name": "grid03", "x": 10.0, "y": 30.0},
    {"name": "grid04", "x": 30.0, "y": 20.0},
]
SYNTHETIC_LABELS = {
    "names": ["label01", "label02", "label03", "label04"],
    "canonical_labels": ["Fp1", "Fp2", "C3", "C4"],
}

print("3D example rows:", SYNTHETIC_3D_ROWS)
print("2D example rows:", SYNTHETIC_2D_ROWS)
print("Canonical-label example:", SYNTHETIC_LABELS)

## 3. In-memory CSV, TSV, and JSON loaders

Real users may supply files with these schemas. To keep this notebook self-contained and non-reconstructive, the examples parse strings held only in memory.

In [ ]:
import csv


def load_delimited_text(text: str, delimiter: str) -> list[dict[str, object]]:
    rows = list(csv.DictReader(io.StringIO(text), delimiter=delimiter))
    if not rows or not {"name", "x", "y"}.issubset(rows[0]):
        raise ValueError("Expected name, x, and y columns")
    axes = ("x", "y", "z") if "z" in rows[0] else ("x", "y")
    return [
        {"name": row["name"], **{axis: float(row[axis]) for axis in axes}}
        for row in rows
    ]


def load_json_text(text: str) -> tuple[list[str], np.ndarray, str]:
    payload = json.loads(text)
    required = {"ch_names", "ch_pos"}
    if not required.issubset(payload):
        raise ValueError("Expected ch_names and ch_pos")
    names = list(map(str, payload["ch_names"]))
    positions = np.asarray(payload["ch_pos"], dtype=float)
    if positions.ndim != 2 or positions.shape[1] not in (2, 3) or len(names) != len(positions):
        raise ValueError("Names and positions must describe the same 2D or 3D rows")
    return names, positions, str(payload.get("meta_ch_pos_unit", "m"))


csv_text = "name,x,y,z\ndemo01,-31,78,52\ndemo02,30,79,51\n"
tsv_text = "name\tx\ty\ngrid01\t-30\t20\ngrid02\t-10\t30\n"
json_text = json.dumps({
    "ch_names": ["demo01", "demo02"],
    "ch_pos": [[-31, 78, 52], [30, 79, 51]],
    "meta_ch_pos_unit": "mm",
})

print("CSV:", load_delimited_text(csv_text, ","))
print("TSV:", load_delimited_text(tsv_text, "\t"))
print("JSON:", load_json_text(json_text))

## 4. Run every public alignment case



The four core alignment methods are exercised separately:



1. **`isotropic_3d_nearest`**: arbitrary 3D head/device coordinates are centered and isotropically scaled before nearest-slot assignment.

2. **`canonical_3d_nearest`**: coordinates already expressed in the target `standard_1005` frame are assigned directly, without centering or rescaling.

3. **`isotropic_2d_nearest`**: planar channel coordinates are centered and scaled in a shared top-down projection before assignment.

4. **`canonical_labels`**: known Standard 1005 labels bypass geometric estimation and map exactly to their reference slots.



`align_optodes` is an additional input-construction path: it creates S-D pair midpoints and then invokes one of the geometric methods above. Dataset-specific paper anchors and axis corrections remain adapter responsibilities; after an adapter places points in `standard_1005`, the public package uses `canonical_3d_nearest`.

In [ ]:
def rows_to_arrays(rows: list[dict[str, object]]) -> tuple[list[str], np.ndarray]:

    names = [str(row["name"]) for row in rows]

    axes = ("x", "y", "z") if "z" in rows[0] else ("x", "y")

    positions = np.asarray([[row[axis] for axis in axes] for row in rows], dtype=float)

    return names, positions





names_3d, positions_3d = rows_to_arrays(SYNTHETIC_3D_ROWS)

result_isotropic_3d = align(

    positions=positions_3d,

    names=names_3d,

    unit="mm",

    frame="head",

)



reference = load_reference()

canonical_ids = [20, 80, 160, 240]

canonical_positions = np.asarray(reference["positions"])[canonical_ids]

canonical_names = [f"canonical_{index + 1}" for index in range(len(canonical_ids))]

result_canonical_3d = align(

    positions=canonical_positions,

    names=canonical_names,

    unit="m",

    frame="standard_1005",

)



names_2d, positions_2d = rows_to_arrays(SYNTHETIC_2D_ROWS)

result_isotropic_2d = align(

    positions=positions_2d,

    names=names_2d,

    unit="mm",

    frame="schematic",

)



result_labels = align(

    positions=np.zeros((len(SYNTHETIC_LABELS["names"]), 3)),

    names=SYNTHETIC_LABELS["names"],

    canonical_labels=SYNTHETIC_LABELS["canonical_labels"],

)



reference_lookup = dict(zip(reference["labels"], reference["positions"]))

source_positions = np.asarray([reference_lookup["FFC3h"], reference_lookup["FFC4h"]])

detector_positions = np.asarray([reference_lookup["FCC3h"], reference_lookup["FCC4h"]])

result_optodes = align_optodes(

    source_positions=source_positions,

    detector_positions=detector_positions,

    channel_pairs=[(0, 0), (1, 1)],

    channel_names=["S1_D1", "S2_D2"],

    frame="standard_1005",

)



CORE_RESULTS = {

    "isotropic 3D": result_isotropic_3d,

    "canonical-frame 3D": result_canonical_3d,

    "isotropic 2D": result_isotropic_2d,

    "canonical labels": result_labels,

}

WORKFLOW_RESULTS = {**CORE_RESULTS, "source-detector midpoint input": result_optodes}

for workflow, result in WORKFLOW_RESULTS.items():

    print(

        f"{workflow:32} method={result.method:22} "

        f"valid={int(result.valid.sum())}/{len(result.valid)} "

        f"output={result.ch_pos_1005.shape} collisions={len(result.collisions)}"

    )



assert {result.method for result in CORE_RESULTS.values()} == {

    "isotropic_3d_nearest",

    "canonical_3d_nearest",

    "isotropic_2d_nearest",

    "canonical_labels",

}

## 5. Validate inputs and inspect preprocessing

The package converts units to metres, aligns coordinates into the reference frame, assigns the nearest Standard 1005 slot, and reports validity, distance, and physical-channel collisions. It never modifies the source input.

In [ ]:
def validate_rows(rows: list[dict[str, object]], dimensions: int) -> None:

    required = {"name", "x", "y"} | ({"z"} if dimensions == 3 else set())

    if not rows:

        raise ValueError("At least one row is required")

    if any(not required.issubset(row) for row in rows):

        raise ValueError(f"Every row must contain {sorted(required)}")

    names = [str(row["name"]) for row in rows]

    if len(names) != len(set(names)):

        raise ValueError("Channel names must be unique")

    _, positions = rows_to_arrays(rows)

    if not np.isfinite(positions).all():

        raise ValueError("Example coordinates must be finite")





validate_rows(SYNTHETIC_3D_ROWS, dimensions=3)

validate_rows(SYNTHETIC_2D_ROWS, dimensions=2)



before_mm = positions_3d.copy()

after_m = result_isotropic_3d.ch_pos_1005

print("Input unchanged:", np.array_equal(before_mm, positions_3d))

print("Before:", before_mm.shape, "synthetic millimetre coordinates")

print("After: ", after_m.shape, "Standard 1005 coordinates in metres")

print("Assigned labels:", result_isotropic_3d.slot_labels)

## 6. Pre-rendered study results

The next images are the only public artifacts derived from the 12 study montage workflows. They are displayed as final raster QC outputs, not loaded as geometry.

In [ ]:
def show_image(path: Path, title: str, *, figsize: tuple[float, float] = (14, 7)) -> None:
    if not path.is_file():
        print(f"Missing rendered image: {path}")
        return
    image = plt.imread(path)
    figure, axis = plt.subplots(figsize=figsize)
    axis.imshow(image)
    axis.set_title(title)
    axis.axis("off")
    plt.show()


show_image(
    FINAL_RESULTS / "montage_pretraining_overview.png",
    "All 12 pretraining datasets after harmonization",
    figsize=(16, 9),
)
show_image(
    FINAL_RESULTS / "montage_alignment_examples.png",
    "Representative alignment strategies",
    figsize=(16, 7),
)

In [ ]:
missing_images = [
    dataset["image"]
    for dataset in DATASETS
    if not (FINAL_RESULTS / dataset["image"]).is_file()
]
if missing_images:
    print("Missing dataset images:", missing_images)
else:
    figure, axes = plt.subplots(4, 3, figsize=(18, 22), constrained_layout=True)
    for axis, dataset in zip(axes.flat, DATASETS):
        image = plt.imread(FINAL_RESULTS / dataset["image"])
        axis.imshow(image)
        axis.set_title(f"{dataset['name']}\n{dataset['strategy']}", fontsize=11)
        axis.axis("off")
    figure.suptitle("Pretraining montage QC gallery", fontsize=18)
    plt.show()

## 7. Public-artifact and schema checks

These checks fail clearly when a rendered image is missing or when a synthetic example no longer matches the documented schema. They do not search for or require any private source file.

In [ ]:
required_dataset_keys = {

    "name",

    "source_format",

    "geometry",

    "strategy",

    "public_input_schema",

    "image",

}

assert len(DATASETS) == 12

assert len({dataset["name"] for dataset in DATASETS}) == 12

assert all(required_dataset_keys == set(dataset) for dataset in DATASETS)

assert not missing_images, f"Missing rendered images: {missing_images}"

assert all(result.valid.all() for result in WORKFLOW_RESULTS.values())

assert all(result.ch_pos_1005.shape[1] == 3 for result in WORKFLOW_RESULTS.values())

assert len(CORE_RESULTS) == 4

assert len(WORKFLOW_RESULTS) == 5

print("Metadata schema: valid")

print("Rendered dataset images: 12/12")

print("Core alignment methods: 4/4 valid")

print("Optode midpoint input path: valid")

## 8. Summary dashboard

`Runnable here` means that this notebook demonstrates the public input class with synthetic data. It does **not** mean that the original third-party dataset geometry is distributed.

In [ ]:
dashboard_rows = []

for dataset in DATASETS:

    strategy = dataset["strategy"]

    if "canonical labels" in strategy:

        demonstrated_by = "canonical labels"

    elif "paper anchor" in strategy:

        demonstrated_by = "canonical-frame 3D"

    elif "axis correction" in strategy or "isotropic 3D" in strategy:

        demonstrated_by = "isotropic 3D"

    elif "optodes" in dataset["geometry"]:

        demonstrated_by = "source-detector midpoint input"

    else:

        demonstrated_by = "isotropic 2D"

    dashboard_rows.append({

        "dataset": dataset["name"],

        "format": dataset["source_format"],

        "geometry": dataset["geometry"],

        "strategy": strategy,

        "public schema": dataset["public_input_schema"],

        "runnable here": demonstrated_by,

        "QC image": "available" if (FINAL_RESULTS / dataset["image"]).is_file() else "missing",

    })



header = ("dataset", "format", "strategy", "runnable here", "QC image")

widths = {key: max(len(key), max(len(str(row[key])) for row in dashboard_rows)) for key in header}

print(" | ".join(key.ljust(widths[key]) for key in header))

print("-+-".join("-" * widths[key] for key in header))

for row in dashboard_rows:

    print(" | ".join(str(row[key]).ljust(widths[key]) for key in header))

## Use with bundled examples or your own data

- Start with [`../example_test_input/README.md`](../example_test_input/README.md) to load the packaged synthetic JSON, TSV, TXT, NPZ, BIDS fNIRS, and SNIRF scenarios in the UI or CLI.
- Use `load_snirf(path)` when your own SNIRF file contains valid embedded geometry.
- Use `MtSi1005.align(positions=..., names=..., unit=...)` or the functional `align(...)` API for channel midpoint arrays.
- Use `MtSi1005.align_optodes(...)` or `align_optodes(...)` for source/detector arrays and known S-D pair indices.
- Use `canonical_labels=...` when Standard 1005 labels are already known.
- Launch `mtsi1005-ui` for manual source/detector placement when no coordinate file is available.

Dataset/vendor parsing and paper-derived assumptions remain outside the generic package. The bundled scenarios are synthetic; for other inputs, only use geometry that you are licensed and permitted to process or distribute.